# Qwen3.5-2B Exam Generator — Unsloth Fine-Tuning & GGUF Export

This notebook distills **Exam Generator** domain expertise (`<<<STUDENT_SHEET>>>` + `<<<ANSWER_KEY>>>` structured exam synthesis with misconception-driven distractors, Bloom-aligned stems, and solve-from-scratch answer keys) into **`unsloth/Qwen3.5-2B`**.

### Configuration Summary
- **Base Model:** `unsloth/Qwen3.5-2B` (`load_in_4bit=False`, bf16/fp16 LoRA — ~5 GB VRAM on T4)
- **Context Length:** `max_seq_length = 3072`
- **Target Hardware:** Google Colab Free **T4 (15 GB VRAM)**, also compatible with **L4 / A100**
- **Training Method:** `SFTTrainer` + `SFTConfig` + `train_on_responses_only` (`enable_thinking=False`)
- **Exports:** LoRA adapter (`qwen3.5_2b_exam_lora`), Ollama-compatible `Q4_K_M` GGUF (`qwen3.5_2b_exam_gguf`, converted and tensor-checked with the llama.cpp revision pinned by Ollama 0.40.16), and an Ollama `Modelfile`

## Step 1 — Setup & Install

Installs modern **Unsloth** with Qwen3.5-compatible `transformers >= 5.2.0` and `trl >= 0.22.2` using `uv` / `pip`, then verifies GPU availability, VRAM, and `bf16` support.

In [1]:
import importlib.metadata
import importlib.util
import os
import shutil
import subprocess
import sys


try:
    initial_numpy_version = importlib.metadata.version("numpy")
except importlib.metadata.PackageNotFoundError:
    initial_numpy_version = None
numpy_pin = [f"numpy=={initial_numpy_version}"] if initial_numpy_version else []

def run_cmd(cmd: list[str]) -> None:
    print("+", " ".join(cmd))
    subprocess.check_call(cmd)


if "COLAB_" in "".join(os.environ.keys()) and shutil.which("nvidia-smi") is None:
    raise RuntimeError(
        "No NVIDIA GPU attached to this Colab runtime! "
        "Click Runtime -> Change runtime type -> Hardware accelerator: T4 GPU -> Save, then re-run."
    )

run_cmd([sys.executable, "-m", "pip", "install", "-q", "uv"])

if "COLAB_" in "".join(os.environ.keys()):
    # 1. Verify preinstalled PyTorch/TorchVision/TorchAudio match the same CUDA build
    torch_check = subprocess.run(
        [sys.executable, "-c", "import torch, torchvision, torchaudio"],
        capture_output=True,
        text=True,
    )
    repaired_torch = False
    if torch_check.returncode != 0:
        print("Detected PyTorch/TorchAudio CUDA mismatch — aligning wheels...")
        repaired_torch = True
        ok = subprocess.run([
            "uv", "pip", "install", "--system", "-q", "--force-reinstall",
            "--torch-backend=auto", "torch", "torchvision", "torchaudio", *numpy_pin,
        ]).returncode == 0
        if (
            not ok
            or subprocess.run(
                [sys.executable, "-c", "import torch, torchvision, torchaudio"],
                capture_output=True,
            ).returncode != 0
        ):
            run_cmd([
                "uv", "pip", "install", "--system", "-q", "--force-reinstall",
                "--index-url", "https://download.pytorch.org/whl/cu130",
                "torch", "torchvision", "torchaudio", *numpy_pin,
            ])
            torch_check = subprocess.run(
                [sys.executable, "-c", "import torch, torchvision, torchaudio"],
                capture_output=True,
                text=True,
            )
            if torch_check.returncode != 0:
                raise RuntimeError(
                    "PyTorch/TorchVision/TorchAudio still fail to import after repair: "
                    f"{torch_check.stderr.strip()}"
                )

    # 2. Install/upgrade non-PyTorch deps normally (never triggers a partial torch upgrade)
    run_cmd([
        "uv", "pip", "install", "--system", "-q", "--upgrade",
        *numpy_pin,
        "sentencepiece", "protobuf", "datasets>=3.4.1,<4.4.0",
        "huggingface_hub>=1.3.0", "tokenizers>=0.22.2", "safetensors",
        "hf_transfer", "tyro", "msgspec", "httpx", "einops",
        "regex", "pyyaml", "packaging", "psutil", "tqdm", "filelock", "typer-slim",
    ])

    # 3. Install/upgrade PyTorch-bound packages with --no-deps so torch stays aligned with torchaudio
    torch_deps = [
        "bitsandbytes", "accelerate", "peft", "cut_cross_entropy",
        "unsloth_zoo", "unsloth", "transformers>=5.2.0", "trl>=0.22.2",
    ]
    if importlib.util.find_spec("triton") is None:
        torch_deps.append("triton")
    run_cmd(["uv", "pip", "install", "--system", "-q", "--upgrade", "--no-deps", *torch_deps])

    if repaired_torch and "torch" in sys.modules:
        raise RuntimeError(
            "PyTorch was replaced after this runtime had imported it. "
            "Restart the Colab runtime, then rerun the notebook from Step 1."
        )
else:
    uv_install = ["uv", "pip", "install", "-q"]
    if sys.prefix == sys.base_prefix:
        uv_install.append("--system")
    run_cmd([
        *uv_install,
        *numpy_pin,
        "unsloth", "transformers>=5.2.0", "trl>=0.22.2",
        "datasets>=3.4.1", "sentencepiece", "protobuf", "hf_transfer",
    ])

loaded_numpy = sys.modules.get("numpy")
if loaded_numpy is not None:
    loaded_numpy_version = getattr(loaded_numpy, "__version__", None)
    installed_numpy_version = importlib.metadata.version("numpy")
    if loaded_numpy_version != installed_numpy_version:
        raise RuntimeError(
            f"NumPy changed during installation (loaded={loaded_numpy_version}, "
            f"installed={installed_numpy_version}). Restart the Colab runtime, "
            "then rerun the notebook from Step 1 before importing packages."
        )

+ /usr/bin/python3 -m pip install -q uv
+ uv pip install --system -q --upgrade numpy==2.5.4 sentencepiece protobuf datasets>=3.4.1,<4.4.0 huggingface_hub>=1.3.0 tokenizers>=0.22.2 safetensors hf_transfer tyro msgspec httpx einops regex pyyaml packaging psutil tqdm filelock typer-slim
+ uv pip install --system -q --upgrade --no-deps bitsandbytes accelerate peft cut_cross_entropy unsloth_zoo unsloth transformers>=5.2.0 trl>=0.22.2


In [2]:
import unsloth
import torch
import transformers
import trl
import peft

print("=== Package Versions ===")
print(f"PyTorch      : {torch.__version__}")
print(f"Transformers : {transformers.__version__}")
print(f"TRL          : {trl.__version__}")
print(f"PEFT         : {peft.__version__}")
print(f"Unsloth      : {getattr(unsloth, '__version__', 'installed')}")

print("\n=== GPU & Precision Check ===")
assert torch.cuda.is_available(), (
    "CUDA GPU is required for fine-tuning. "
    "In Google Colab: click Runtime -> Change runtime type -> T4 GPU -> Save, then re-run Step 1 & 2."
)
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
gpu_props = torch.cuda.get_device_properties(0)
vram_gb = gpu_props.total_memory / (1024 ** 3)
bf16_ok = torch.cuda.is_bf16_supported()
print(f"Active GPU   : {gpu_props.name}")
print(f"Total VRAM   : {vram_gb:.2f} GB")
print(f"BF16 Support : {bf16_ok} (using {'bf16' if bf16_ok else 'fp16'})")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/lib/python3.13/dist-packages/transformers/configuration_utils.py:726: FutureWarning: `rope_scaling` is deprecated and will be removed in a future version. Set `rope_parameters` instead, with `rope_type` and `rope_theta` included.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (
/usr/local/lib/python3.13/dist-packages/unsloth_zoo/temporary_patches/gemma.py:279: FutureWarning: `Gemma3ProcessorKwargs` is deprecated and will be removed in version 5.29.0. Use `ProcessingKwargs` instead.
  from transformers.models.gemma3.processing_gemma3 import Gemma3ProcessorKwargs


🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: Could not find `steps_per_generation` in async_grpo_trainer
Unsloth: Could not find `generation_batch_size` in async_grpo_trainer
=== Package Versions ===
PyTorch      : 2.14.1+cu130
Transformers : 5.23.0
TRL          : 1.18.0
PEFT         : 0.21.3
Unsloth      : 2026.11.4

=== GPU & Precision Check ===
Tesla T4, 15360 MiB, 580.82.07
Active GPU   : Tesla T4
Total VRAM   : 14.56 GB
BF16 Support : False (using fp16)


## Step 2 — Load `unsloth/Qwen3.5-2B` (bf16/fp16 LoRA)

Per Unsloth's official Qwen3.5 recommendation, we set `load_in_4bit=False` to avoid 4-bit QLoRA quantization variance. At 2B parameters in fp16/bf16 LoRA, the model uses roughly ~5 GB VRAM, leaving generous headroom on a 15 GB T4 for `max_seq_length = 3072`.

In [3]:
import torch
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Qwen3.5-2B"
max_seq_length = 3072
dtype = None          # Auto-detect: fp16 on T4, bf16 on L4/A100
load_in_4bit = False  # Avoid 4-bit quantization variance on Qwen3.5-2B (~5 GB VRAM)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

model.print_trainable_parameters()
allocated_gb = torch.cuda.memory_allocated() / (1024 ** 3)
reserved_gb = torch.cuda.memory_reserved() / (1024 ** 3)
print(f"VRAM allocated: {allocated_gb:.2f} GB | reserved: {reserved_gb:.2f} GB")

/usr/local/lib/python3.13/dist-packages/unsloth_zoo/temporary_patches/gemma.py:279: FutureWarning: `Gemma3ProcessorKwargs` is deprecated and will be removed in version 5.29.0. Use `ProcessingKwargs` instead.
  from transformers.models.gemma3.processing_gemma3 import Gemma3ProcessorKwargs


==((====))==  Unsloth 2026.11.4: Fast Qwen3_5 patching. Transformers: 5.23.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


model.safetensors.index.json:   0%|          | 0.00/64.5k [00:00<?, ?B/s]

Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3303: FutureWarning: `is_gradient_checkpointing` is deprecated and will be removed in v6. Use `is_activation_checkpointing` instead.
  warnings.warn(


Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.


trainable params: 10,911,744 || all params: 2,224,153,408 || trainable%: 0.4906
VRAM allocated: 4.17 GB | reserved: 4.25 GB


## Step 3 — Load & Format `dataset_train.jsonl` and `dataset_val.jsonl`

Supports three ways to supply the dataset:
1. Place `dataset_train.jsonl` and `dataset_val.jsonl` in the working directory or Google Drive (`DRIVE_DATA_DIR`).
2. Supply `dataset_exam_generator.jsonl` (auto-splits 90% train / 10% val with `seed=3407`).
3. Upload interactively via Colab's file picker if none of the files are present yet.

In [4]:
import sys
from pathlib import Path
from datasets import load_dataset

# Optional: set USE_GOOGLE_DRIVE = True if your JSONL files live in Google Drive
USE_GOOGLE_DRIVE = False
DRIVE_DATA_DIR = Path("/content/drive/MyDrive/qwen_exam_generator")

if USE_GOOGLE_DRIVE and "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")


def _find_file(filename: str) -> Path | None:
    candidates = [
        Path(filename),
        Path("/content") / filename,
        DRIVE_DATA_DIR / filename,
    ]
    for path in candidates:
        if path.is_file():
            return path
    return None


train_file = _find_file("dataset_train.jsonl")
val_file = _find_file("dataset_val.jsonl")
combined_file = _find_file("dataset_exam_generator.jsonl")

if (train_file is None or val_file is None) and combined_file is None and "google.colab" in sys.modules:
    from google.colab import files
    print("Upload dataset_train.jsonl & dataset_val.jsonl (or dataset_exam_generator.jsonl):")
    uploaded = files.upload()
    train_file = _find_file("dataset_train.jsonl")
    val_file = _find_file("dataset_val.jsonl")
    combined_file = _find_file("dataset_exam_generator.jsonl")

if train_file is not None and val_file is not None:
    print(f"Loading train={train_file} and val={val_file}")
    raw_ds = load_dataset(
        "json",
        data_files={"train": str(train_file), "validation": str(val_file)},
    )
    train_raw = raw_ds["train"]
    val_raw = raw_ds["validation"]
elif combined_file is not None:
    print(f"Loading {combined_file} with automatic 90/10 train/val split...")
    full_ds = load_dataset("json", data_files=str(combined_file), split="train")
    split_ds = full_ds.train_test_split(test_size=0.1, seed=3407)
    train_raw = split_ds["train"]
    val_raw = split_ds["test"]
else:
    raise FileNotFoundError(
        "Dataset not found. Provide dataset_train.jsonl + dataset_val.jsonl "
        "(or dataset_exam_generator.jsonl) in the current directory or DRIVE_DATA_DIR."
    )

print(f"Raw split sizes -> train: {len(train_raw)}, val: {len(val_raw)}")

Upload dataset_train.jsonl & dataset_val.jsonl (or dataset_exam_generator.jsonl):


Saving dataset_exam_generator.jsonl to dataset_exam_generator.jsonl
Saving dataset_train.jsonl to dataset_train.jsonl
Saving dataset_val.jsonl to dataset_val.jsonl
Loading train=dataset_train.jsonl and val=dataset_val.jsonl


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Raw split sizes -> train: 86, val: 10


In [5]:
# Format conversations using Qwen3.5 ChatML template with enable_thinking=False
raw_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)


def _extract_messages(example: dict) -> list[dict]:
    if "messages" in example and example["messages"]:
        return example["messages"]
    if "conversations" in example and example["conversations"]:
        role_map = {"human": "user", "gpt": "assistant", "system": "system"}
        return [
            {
                "role": role_map.get(turn.get("from", turn.get("role", "")), turn.get("role", "user")),
                "content": turn.get("value", turn.get("content", "")),
            }
            for turn in example["conversations"]
        ]
    raise KeyError("Record must contain a 'messages' (or 'conversations') field.")


def apply_chat_template_safe(
    convo: list[dict],
    add_generation_prompt: bool = False,
    enable_thinking: bool = False,
) -> str:
    try:
        return tokenizer.apply_chat_template(
            convo,
            tokenize=False,
            add_generation_prompt=add_generation_prompt,
            enable_thinking=enable_thinking,
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            convo,
            tokenize=False,
            add_generation_prompt=add_generation_prompt,
        )


def formatting_prompts_func(examples: dict) -> dict:
    keys = list(examples.keys())
    n_rows = len(examples[keys[0]])
    texts = []
    for i in range(n_rows):
        row = {k: examples[k][i] for k in keys}
        convo = _extract_messages(row)
        texts.append(
            apply_chat_template_safe(
                convo,
                add_generation_prompt=False,
                enable_thinking=False,
            )
        )
    return {"text": texts}


train_ds = train_raw.map(formatting_prompts_func, batched=True)
val_ds = val_raw.map(formatting_prompts_func, batched=True)


def print_length_stats(name: str, ds) -> None:
    lengths = [len(raw_tokenizer.encode(t, add_special_tokens=False)) for t in ds["text"]]
    over_limit = sum(1 for ln in lengths if ln > max_seq_length)
    avg_len = sum(lengths) / max(1, len(lengths))
    print(
        f"[{name}] count={len(lengths)} | min={min(lengths)} | "
        f"avg={avg_len:.1f} | max={max(lengths)} | over_{max_seq_length}={over_limit}"
    )


print_length_stats("train", train_ds)
print_length_stats("val", val_ds)
print("\nPreview repr(train_ds[0]['text'][:600]):")
print(repr(train_ds[0]["text"][:600]))

Map:   0%|          | 0/86 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

[train] count=86 | min=1303 | avg=2337.3 | max=3243 | over_3072=5
[val] count=10 | min=1227 | avg=2340.8 | max=3176 | over_3072=3

Preview repr(train_ds[0]['text'][:600]):
'<|im_start|>system\nYou are an expert exam designer. Create rigorous exams matching the requested topic, audience level, format, count, and language (Indonesian or English).\n\nRules:\n- MCQ counts: 4 options (A-D) for SD/SMP; 5 options (A-E) for SMA/SMK/Kuliah/University.\n- Stems & stimuli: Self-contained with all constants/data. Ground C3-C6 items in authentic stimuli (data, code, scenarios). Bold and capitalize negative markers (**BUKAN**, **KECUALI**, **NOT**, **EXCEPT**); no double negatives.\n- Distractors: Misconception-driven, same-scope, length- and tone-calibrated. No "All/None of the abo'


## Step 4 — Train with `SFTTrainer` + `SFTConfig` + `train_on_responses_only`

We dynamically inspect `train_ds[0]["text"]` to match the exact assistant prefix (whether `<|im_start|>assistant\n<think>\n\n</think>\n\n` or `<|im_start|>assistant\n`) so `train_on_responses_only` never masks out all labels, and we verify non-empty supervised tokens before training starts.

In [6]:
import inspect
import re
import time
import torch
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

# Dynamically detect exact instruction and response delimiters from formatted text
sample_text = train_ds[0]["text"]
instruction_part = "<|im_start|>user\n"

resp_match = re.search(
    r"<\|im_start\|>assistant\n(?:<think>\s*</think>\s*)?",
    sample_text,
)
if resp_match:
    response_part = resp_match.group(0)
else:
    response_part = "<|im_start|>assistant\n"

print(f"Detected instruction_part : {repr(instruction_part)}")
print(f"Detected response_part    : {repr(response_part)}")

use_bf16 = torch.cuda.is_bf16_supported()

# Build SFTConfig & SFTTrainer kwargs safely across TRL versions
sft_config_params = inspect.signature(SFTConfig.__init__).parameters
sft_trainer_params = inspect.signature(SFTTrainer.__init__).parameters

cfg_kwargs = {
    "output_dir": "outputs",
    "per_device_train_batch_size": 1,
    "per_device_eval_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "num_train_epochs": 3,
    "learning_rate": 2e-4,
    "lr_scheduler_type": "cosine",
    "warmup_steps": 5,
    "eval_steps": 10,
    "save_strategy": "steps",
    "save_steps": 20,
    "save_total_limit": 2,
    "logging_steps": 1,
    "optim": "adamw_8bit",
    "weight_decay": 0.01,
    "fp16": not use_bf16,
    "bf16": use_bf16,
    "seed": 3407,
    "report_to": "none",
    "load_best_model_at_end": True,
    "metric_for_best_model": "eval_loss",
}

if "eval_strategy" in sft_config_params:
    cfg_kwargs["eval_strategy"] = "steps"
elif "evaluation_strategy" in sft_config_params:
    cfg_kwargs["evaluation_strategy"] = "steps"

if "max_length" in sft_config_params:
    cfg_kwargs["max_length"] = max_seq_length
elif "max_seq_length" in sft_config_params:
    cfg_kwargs["max_seq_length"] = max_seq_length

if "dataset_text_field" in sft_config_params:
    cfg_kwargs["dataset_text_field"] = "text"
if "dataset_num_proc" in sft_config_params:
    cfg_kwargs["dataset_num_proc"] = 2
if "packing" in sft_config_params:
    cfg_kwargs["packing"] = False

sft_config = SFTConfig(**cfg_kwargs)

trainer_kwargs = {
    "model": model,
    "train_dataset": train_ds,
    "eval_dataset": val_ds,
    "args": sft_config,
}

if "processing_class" in sft_trainer_params:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in sft_trainer_params:
    trainer_kwargs["tokenizer"] = tokenizer

if "dataset_text_field" in sft_trainer_params and "dataset_text_field" not in cfg_kwargs:
    trainer_kwargs["dataset_text_field"] = "text"
if "max_seq_length" in sft_trainer_params and "max_seq_length" not in cfg_kwargs and "max_length" not in cfg_kwargs:
    trainer_kwargs["max_seq_length"] = max_seq_length
if "dataset_num_proc" in sft_trainer_params and "dataset_num_proc" not in cfg_kwargs:
    trainer_kwargs["dataset_num_proc"] = 2
if "packing" in sft_trainer_params and "packing" not in cfg_kwargs:
    trainer_kwargs["packing"] = False

trainer = SFTTrainer(**trainer_kwargs)
trainer = train_on_responses_only(
    trainer,
    instruction_part=instruction_part,
    response_part=response_part,
)

# Critical pre-training verification: ensure response labels are NOT all masked (-100)
check_count = min(5, len(trainer.train_dataset))
for idx in range(check_count):
    sample_row = trainer.train_dataset[idx]
    labels = sample_row["labels"]
    unmasked = sum(1 for x in labels if x != -100)
    masked = sum(1 for x in labels if x == -100)
    assert unmasked > 0, (
        f"All labels were masked (-100) in row {idx}! "
        f"Check response_part={repr(response_part)} against formatted text."
    )
    if idx == 0:
        print(f"Label check row 0: total={len(labels)} | masked(-100)={masked} | supervised={unmasked}")
        unmasked_ids = [tok for tok, lbl in zip(sample_row["input_ids"], labels) if lbl != -100]
        print("Supervised target preview:")
        print(repr(raw_tokenizer.decode(unmasked_ids[:80])))

Detected instruction_part : '<|im_start|>user\n'
Detected response_part    : '<|im_start|>assistant\n<think>\n\n</think>\n\n'


/usr/local/lib/python3.13/dist-packages/transformers/training_args.py:1541: FutureWarning: `gradient_checkpointing` is deprecated and will be removed in v6. Use `activation_checkpointing` instead.
  warnings.warn(f"`{old}` is deprecated and will be removed in v6. Use `{new}` instead.", FutureWarning)
`causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/86 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/10 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map:   0%|          | 0/86 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Label check row 0: total=1587 | masked(-100)=442 | supervised=1145
Supervised target preview:
'<<<STUDENT_SHEET>>>\n**UJIAN IPA SD — KELAS 5**\n**Topik:** Sistem Pencernaan dan Peredaran Darah Manusia\n**Petunjuk:** Pilihlah satu jawaban yang tepat untuk soal 1–2, pilihlah lebih dari satu pernyataan yang benar untuk soal 3 (Pilihan Ganda Kompleks: skor penuh jika memilih semua pernyataan benar tanpa'


In [7]:
# Run SFT training and evaluate on validation set
start_gpu_mem = round(torch.cuda.max_memory_reserved() / (1024 ** 3), 3)
total_gpu_mem = round(torch.cuda.get_device_properties(0).total_memory / (1024 ** 3), 3)

t0 = time.time()
trainer_stats = trainer.train()
train_elapsed = time.time() - t0

eval_metrics = trainer.evaluate()
peak_gpu_mem = round(torch.cuda.max_memory_reserved() / (1024 ** 3), 3)

print("\n=== Training & Evaluation Summary ===")
print(f"Train runtime    : {train_elapsed:.1f}s ({trainer_stats.metrics.get('train_runtime', train_elapsed):.1f}s)")
print(f"Final train loss : {trainer_stats.training_loss:.4f}")
print(f"Eval loss        : {eval_metrics.get('eval_loss', float('nan')):.4f}")
print(f"VRAM start       : {start_gpu_mem:.2f} GB / {total_gpu_mem:.2f} GB")
print(f"VRAM peak        : {peak_gpu_mem:.2f} GB / {total_gpu_mem:.2f} GB ({peak_gpu_mem / total_gpu_mem * 100:.1f}%)")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 86 | Num Epochs = 3 | Total steps = 33
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 10,911,744 of 2,224,153,408 (0.49% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
10,1.219237,1.135901


Filter:   0%|          | 0/10 [00:00<?, ? examples/s]

Unsloth: Not an error, but `num_items_in_batch` could not be counted for Qwen3_5ForConditionalGeneration's labels, so each micro-batch loss is a mean.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


Step,Training Loss,Validation Loss
10,1.219237,1.135901
20,0.945354,1.133713
30,1.042600,1.127574
33,0.912884,1.127434


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-20/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-33/tokenizer_config.json.


Training Loss,Validation Loss,Step
0.912884,1.127434,33



=== Training & Evaluation Summary ===
Train runtime    : 992.7s (991.1s)
Final train loss : 1.0932
Eval loss        : 1.1274
VRAM start       : 5.12 GB / 14.56 GB
VRAM peak        : 5.12 GB / 14.56 GB (35.2%)


## Step 5 — Inference Verification (Aligned with `schema.py`)

Tests the fine-tuned model using the exact `CANONICAL_SYSTEM_PROMPT` from `schema.py`, Qwen3.5 non-thinking sampling (`enable_thinking=False`, `temperature=0.6`, `top_p=0.85`, `top_k=20`, `repetition_penalty=1.05`), `<think>` stripping, and `<<<STUDENT_SHEET>>>` / `<<<ANSWER_KEY>>>` extraction on:
1. **Indonesian SMP** prompt (4 options: `A`–`D`)
2. **Indonesian Kuliah** prompt (5 options: `A`–`E` + Essay)

In [9]:
import re
from unsloth import FastLanguageModel

CANONICAL_SYSTEM_PROMPT = """You are an expert exam designer. Create rigorous exams matching the requested topic, audience level, format, count, and language (Indonesian or English).

Rules:
- MCQ counts: 4 options (A-D) for SD/SMP; 5 options (A-E) for SMA/SMK/Kuliah/University.
- Stems & stimuli: Self-contained with all constants/data. Ground C3-C6 items in authentic stimuli (data, code, scenarios). Bold and capitalize negative markers (**BUKAN**, **KECUALI**, **NOT**, **EXCEPT**); no double negatives.
- Distractors: Misconception-driven, same-scope, length- and tone-calibrated. No "All/None of the above" or "Semua benar/salah". Order numeric/sequential options monotonically.
- Complex MCQ & Essay/Case: Complex MCQs need independently verifiable statements and an explicit scoring/selection rule. Essays/cases need concrete tasks (predict, trace, diagnose, fix, defend trade-offs) with per-part point weights (e.g., [10 poin] or [10 pts]).
- Audit: Solve from scratch, balance MCQ key letters, and exclude answers, hints, and Bloom labels from the student sheet unless requested.

Output ONLY these two tagged sections:
<<<STUDENT_SHEET>>>
Header, instructions, and numbered questions only.
<<</STUDENT_SHEET>>>
<<<ANSWER_KEY>>>
MCQ: correct letter, why right, and one line per distractor naming its misconception. Essay/Case: model solution and point breakdown summing to each sub-part weight.
<<</ANSWER_KEY>>>"""

_THINK_RE = re.compile(r"<think>.*?</think>\s*", re.DOTALL | re.IGNORECASE)
_STUDENT_RE = re.compile(r"<<<STUDENT_SHEET>>>(.*?)<<</STUDENT_SHEET>>>", re.DOTALL)
_ANSWER_RE = re.compile(r"<<<ANSWER_KEY>>>(.*?)<<</ANSWER_KEY>>>", re.DOTALL)


def strip_thinking(text: str) -> str:
    cleaned = _THINK_RE.sub("", text or "")
    cleaned = re.sub(r"</?think>", "", cleaned, flags=re.IGNORECASE)
    return cleaned.strip()


def parse_exam_output(text: str) -> dict[str, str]:
    raw_clean = strip_thinking(text)
    s_match = _STUDENT_RE.search(raw_clean)
    a_match = _ANSWER_RE.search(raw_clean)
    return {
        "student_sheet": s_match.group(1).strip() if s_match else "",
        "answer_key": a_match.group(1).strip() if a_match else "",
        "raw_clean": raw_clean,
    }


FastLanguageModel.for_inference(model)


def generate_exam(user_prompt: str, max_new_tokens: int = 3072) -> dict[str, str]:
    messages = [
        {"role": "system", "content": CANONICAL_SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]
    prompt_text = apply_chat_template_safe(
        messages,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = raw_tokenizer(prompt_text, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.6,
            top_p=0.85,
            top_k=20,
            repetition_penalty=1.05,
            pad_token_id=raw_tokenizer.eos_token_id,
        )
    gen_ids = outputs[0][inputs["input_ids"].shape[-1]:]
    decoded = raw_tokenizer.decode(gen_ids, skip_special_tokens=True)
    result = parse_exam_output(decoded)
    generated_tokens = int(gen_ids.numel())
    eos_id = raw_tokenizer.eos_token_id
    ended_with_eos = bool(
        generated_tokens and eos_id is not None and int(gen_ids[-1]) == eos_id
    )
    result.update({
        "generated_tokens": str(generated_tokens),
        "ended_with_eos": str(ended_with_eos),
        "hit_token_limit": str(generated_tokens >= max_new_tokens and not ended_with_eos),
    })
    return result


def require_complete_exam(label: str, result: dict[str, str]) -> None:
    if result["student_sheet"] and result["answer_key"]:
        return
    raw = result["raw_clean"]
    print(
        f"[{label}] incomplete output: generated_tokens={result['generated_tokens']}, "
        f"ended_with_eos={result['ended_with_eos']}, hit_token_limit={result['hit_token_limit']}"
    )
    print(
        "Markers: student_start=", "<<<STUDENT_SHEET>>>" in raw,
        "student_end=", "<<</STUDENT_SHEET>>>" in raw,
        "answer_start=", "<<<ANSWER_KEY>>>" in raw,
        "answer_end=", "<<</ANSWER_KEY>>>" in raw,
    )
    print("Raw output tail:", repr(raw[-1800:]))
    raise RuntimeError(f"{label} did not produce both complete tagged sections.")


# Test 1: Indonesian SMP (4-option MCQ: A-D)
smp_prompt = (
    "Buatkan paket soal ujian dengan spesifikasi berikut:\n"
    "- Topik: Tekanan Hidrostatis dan Hukum Archimedes\n"
    "- Jenjang Pendidikan: SMP\n"
    "- Profil Kognitif (Bloom): C1-C4\n"
    "- Format Soal: MCQ\n"
    "- Jumlah Soal: 4\n"
    "- Bahasa: Bahasa Indonesia"
)

print("=== TEST 1: Indonesian SMP (Expected 4 options A-D) ===")
smp_res = generate_exam(smp_prompt)
print(f"Extracted student_sheet chars: {len(smp_res['student_sheet'])} | answer_key chars: {len(smp_res['answer_key'])}")
require_complete_exam("TEST 1 Indonesian SMP", smp_res)
print("\n--- SMP STUDENT SHEET ---\n", smp_res["student_sheet"][:1500])
print("\n--- SMP ANSWER KEY ---\n", smp_res["answer_key"][:1500])

=== TEST 1: Indonesian SMP (Expected 4 options A-D) ===
Extracted student_sheet chars: 1441 | answer_key chars: 4360

--- SMP STUDENT SHEET ---
 # Ujian Fisika SMP — Tekanan Hidrostatis dan Hukum Archimedes
**Petunjuk:** Pilihlah satu jawaban yang paling tepat (A, B, C, atau D) untuk setiap soal.

1. Sebuah batu bermassa **2,5 kg** dimasukkan ke dalam sebuah wadah tertutup berisi air murni. Jika massa jenis air adalah **1,0 g/cm³**, berapakah gaya tekan air pada dasar wadah tersebut?
A. 2,5 N
B. 25 N
C. 98 N
D. 250 N

2. Sebuah balok bermassa **1,5 kg** diletakkan di atas permukaan datar air laut (massa jenis air laut = **1,025 g/cm³**), sedangkan di bawahnya terdapat pelatatan air tawar (massa jenis air tawar = **1,00 g/cm³**) sehingga permukaan kedua zat berada pada kedalaman **1,0 m**. Berapakah besar gaya tekan air laut pada dasar pelatatan air tawar?
A. 980 N
B. 1.020 N
C. 1.070 N
D. 1.120 N

3. Sebuah benda bermassa **100 g** diletakkan di atas pelatatan air tawar (massa jenis ai

In [10]:
# Test 2: Indonesian Kuliah (5-option MCQ A-E + Essay)
kuliah_prompt = (
    "Buatkan paket soal ujian dengan spesifikasi berikut:\n"
    "- Topik: Algoritma Pencarian Heuristik (A* dan Greedy Best-First Search)\n"
    "- Jenjang Pendidikan: Kuliah\n"
    "- Profil Kognitif (Bloom): C3-C6\n"
    "- Format Soal: Mixed (2 MCQ, 1 Essay)\n"
    "- Jumlah Soal: 3\n"
    "- Bahasa: Bahasa Indonesia"
)

print("=== TEST 2: Indonesian Kuliah (Expected 5 options A-E + Essay) ===")
kuliah_res = generate_exam(kuliah_prompt)
print(f"Extracted student_sheet chars: {len(kuliah_res['student_sheet'])} | answer_key chars: {len(kuliah_res['answer_key'])}")
require_complete_exam("TEST 2 Indonesian Kuliah", kuliah_res)
print("\n--- KULIAH STUDENT SHEET ---\n", kuliah_res["student_sheet"][:1800])
print("\n--- KULIAH ANSWER KEY ---\n", kuliah_res["answer_key"][:1800])

=== TEST 2: Indonesian Kuliah (Expected 5 options A-E + Essay) ===
Extracted student_sheet chars: 2157 | answer_key chars: 2308

--- KULIAH STUDENT SHEET ---
 # Ujian Algoritma Pencarian Heuristik (A* dan Greedy Best-First Search)
**Jenjang:** Kuliah | **Format:** Mixed (2 MCQ, 1 Essay) | **Total Soal:** 3 | **Bahasa:** Bahasa Indonesia

1. **MCQ (Pilihlah satu jawaban yang tepat):** A* search algorithm menggunakan fungsi heuristik $h(n)$ untuk mengestimasi jarak dari node $n$ ke tujuan $T$. Jika $f(n) = g(n) + h(n)$ selalu kurang dari atau sama dengan $g(T)$ saat node $n$ berada di jalur menuju $T$, maka algoritma tersebut **BUKAN** menjamin menemukan solusi optimal.
   A. True (A* tidak pernah menemukan solusi optimal jika $h(n)$ tidak aditif)
   B. False (A* selalu menemukan solusi optimal jika $h(n)$ aditif dan $T$ belum tercapai)
   C. True (A* selalu menemukan solusi optimal jika $h(n)$ aditif dan $T$ sudah tercapai)
   D. False (A* selalu menemukan solusi optimal jika $h(n)$ tid

## Step 6 - Export LoRA, Ollama-compatible GGUF (`Q4_K_M`), and `Modelfile`

Saves the LoRA adapter, merges it into a 16-bit Hugging Face checkpoint, then converts and quantizes with the llama.cpp revision pinned by Ollama 0.40.16. The export verifies that separate SSM beta and alpha tensors are present before and after quantization; this avoids the fused-tensor layout that causes Ollama to fail with a missing `ssm_beta.weight`. The generated GGUF and Modelfile are saved in the output directory.

In [11]:
import subprocess
import sys
import tempfile
from pathlib import Path

def run_cmd(cmd: list[str]) -> None:
    print("+", " ".join(cmd))
    subprocess.check_call(cmd)

LORA_DIR = Path("qwen3.5_2b_exam_lora")
GGUF_DIR = Path("qwen3.5_2b_exam_gguf")
LLAMA_CPP_COMMIT = "6c01da6278ab6c69ed67d9124806b50d2b299ba7"
LLAMA_CPP_DIR = Path("/content/llama.cpp-b12177")

# 1. Save LoRA adapters
model.save_pretrained(str(LORA_DIR))
tokenizer.save_pretrained(str(LORA_DIR))
print(f"Saved LoRA adapter to ./{LORA_DIR}/")

# 2. Use the same llama.cpp revision as Ollama 0.40.16.
if not LLAMA_CPP_DIR.exists():
    run_cmd([
        "git", "clone", "--filter=blob:none", "--no-checkout",
        "https://github.com/ggml-org/llama.cpp.git", str(LLAMA_CPP_DIR),
    ])
    run_cmd([
        "git", "-C", str(LLAMA_CPP_DIR), "checkout", "--detach", LLAMA_CPP_COMMIT,
    ])

actual_commit = subprocess.check_output(
    ["git", "-C", str(LLAMA_CPP_DIR), "rev-parse", "HEAD"], text=True
).strip()
if actual_commit != LLAMA_CPP_COMMIT:
    raise RuntimeError(
        f"Expected llama.cpp {LLAMA_CPP_COMMIT}, found {actual_commit} at {LLAMA_CPP_DIR}. "
        "Use a fresh Colab runtime or remove that specific checkout and rerun this cell."
    )

build_dir = LLAMA_CPP_DIR / "build"
run_cmd([
    "cmake", "-S", str(LLAMA_CPP_DIR), "-B", str(build_dir),
    "-DGGML_CUDA=OFF", "-DCMAKE_BUILD_TYPE=Release",
])
run_cmd([
    "cmake", "--build", str(build_dir), "--config", "Release",
    "--target", "llama-quantize", "-j", "2",
])
quantizer = build_dir / "bin" / "llama-quantize"
if not quantizer.is_file():
    raise FileNotFoundError(f"llama-quantize build output was not found: {quantizer}")

sys.path.insert(0, str(LLAMA_CPP_DIR / "gguf-py"))
from gguf import GGUFReader


def verify_ollama_ssm_layout(gguf_path: Path) -> None:
    names = {tensor.name for tensor in GGUFReader(str(gguf_path)).tensors}
    beta_layers = {
        name.split(".")[1]
        for name in names
        if name.startswith("blk.") and name.endswith(".ssm_beta.weight")
    }
    alpha_layers = {
        name.split(".")[1]
        for name in names
        if name.startswith("blk.") and name.endswith(".ssm_alpha.weight")
    }
    if not beta_layers or beta_layers != alpha_layers:
        raise RuntimeError(
            f"GGUF has mismatched SSM beta/alpha tensors: beta={len(beta_layers)}, "
            f"alpha={len(alpha_layers)}. Expected separate tensors for every layer."
        )
    if any(name.endswith(".ssm_beta_alpha.weight") for name in names):
        raise RuntimeError("GGUF contains fused SSM beta/alpha tensors incompatible with Ollama 0.40.16.")
    print(f"Verified separate SSM beta/alpha tensors for {len(beta_layers)} layers in {gguf_path}")


# 3. Merge the adapter and convert/quantize using the pinned converter/runtime pair.
GGUF_DIR.mkdir(parents=True, exist_ok=True)
Q4_GGUF = GGUF_DIR / "Qwen3.5-2B.Q4_K_M.gguf"
with tempfile.TemporaryDirectory(prefix="qwen35-export-") as temp_dir:
    temp_path = Path(temp_dir)
    merged_dir = temp_path / "merged"
    f16_gguf = temp_path / "Qwen3.5-2B.F16.gguf"
    model.save_pretrained_merged(
        str(merged_dir), tokenizer, save_method="merged_16bit"
    )
    run_cmd([
        sys.executable, str(LLAMA_CPP_DIR / "convert_hf_to_gguf.py"),
        str(merged_dir), "--outfile", str(f16_gguf), "--outtype", "f16",
    ])
    verify_ollama_ssm_layout(f16_gguf)
    run_cmd([str(quantizer), str(f16_gguf), str(Q4_GGUF), "Q4_K_M"])

verify_ollama_ssm_layout(Q4_GGUF)
print("Exported Ollama-compatible GGUF:", Q4_GGUF)
gguf_rel_path = "./" + Q4_GGUF.as_posix()

# 4. Generate Modelfile matching training response_part
think_prefill = "<think>\n\n</think>\n\n" if "<think>" in response_part else ""

MODELFILE_TEMPLATE = """FROM __GGUF_PATH__

TEMPLATE \"\"\"{{- if .Messages }}{{- if .System }}<|im_start|>system
{{ .System }}<|im_end|>
{{ end }}{{- range $i, $_ := .Messages }}{{- $last := eq (len (slice $.Messages $i)) 1 -}}{{- if eq .Role "user" }}<|im_start|>user
{{ .Content }}<|im_end|>
{{ else if eq .Role "assistant" }}<|im_start|>assistant
__THINK_PREFILL__{{ .Content }}{{ if not $last }}<|im_end|>
{{ end }}{{- end }}{{- if and (ne .Role "assistant") $last }}<|im_start|>assistant
__THINK_PREFILL__{{ end }}{{- end }}{{- else }}{{- if .System }}<|im_start|>system
{{ .System }}<|im_end|>
{{ end }}{{- if .Prompt }}<|im_start|>user
{{ .Prompt }}<|im_end|>
{{ end }}<|im_start|>assistant
__THINK_PREFILL__{{ end }}{{ .Response }}{{ if .Response }}<|im_end|>{{ end }}\"\"\"

SYSTEM \"\"\"__SYSTEM_PROMPT__\"\"\"

PARAMETER stop "<|im_start|>"
PARAMETER stop "<|im_end|>"
PARAMETER stop "<|endoftext|>"
PARAMETER num_ctx 4096
PARAMETER temperature 0.6
PARAMETER top_p 0.85
PARAMETER top_k 20
PARAMETER repeat_penalty 1.05
"""

modelfile_content = (
    MODELFILE_TEMPLATE
    .replace("__GGUF_PATH__", gguf_rel_path)
    .replace("__THINK_PREFILL__", think_prefill)
    .replace("__SYSTEM_PROMPT__", CANONICAL_SYSTEM_PROMPT)
)

Path("Modelfile").write_text(modelfile_content, encoding="utf-8")
print("Generated ./Modelfile pointing to:", gguf_rel_path)
print("\n--- Modelfile Preview (first 25 lines) ---")
print("\n".join(modelfile_content.splitlines()[:25]))

Unsloth: Restored added_tokens_decoder metadata in qwen3.5_2b_exam_lora/tokenizer_config.json.


Saved LoRA adapter to ./qwen3.5_2b_exam_lora/
+ git clone --filter=blob:none --no-checkout https://github.com/ggml-org/llama.cpp.git /content/llama.cpp-b12177
+ git -C /content/llama.cpp-b12177 checkout --detach 6c01da6278ab6c69ed67d9124806b50d2b299ba7
+ cmake -S /content/llama.cpp-b12177 -B /content/llama.cpp-b12177/build -DGGML_CUDA=OFF -DCMAKE_BUILD_TYPE=Release
+ cmake --build /content/llama.cpp-b12177/build --config Release --target llama-quantize -j 2


Unsloth: Restored added_tokens_decoder metadata in /tmp/qwen35-export-cyft8ill/merged/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...




Unsloth: Copying 1 files from cache to `/tmp/qwen35-export-cyft8ill/merged`: 100%|██████████| 1/1 [00:00<00:00, 1885.08it/s]


Successfully copied all 1 files from cache to `/tmp/qwen35-export-cyft8ill/merged`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.




Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:00<00:00, 2743.17it/s]


Unsloth: Merging weights into 16bit:   0%|          | 0/1 [00:00<?, ?it/s]

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:43<00:00, 103.19s/it]


Unsloth: Merge process complete. Saved to `/tmp/qwen35-export-cyft8ill/merged`
+ /usr/bin/python3 /content/llama.cpp-b12177/convert_hf_to_gguf.py /tmp/qwen35-export-cyft8ill/merged --outfile /tmp/qwen35-export-cyft8ill/Qwen3.5-2B.F16.gguf --outtype f16
Verified separate SSM beta/alpha tensors for 18 layers in /tmp/qwen35-export-cyft8ill/Qwen3.5-2B.F16.gguf
+ /content/llama.cpp-b12177/build/bin/llama-quantize /tmp/qwen35-export-cyft8ill/Qwen3.5-2B.F16.gguf qwen3.5_2b_exam_gguf/Qwen3.5-2B.Q4_K_M.gguf Q4_K_M
Verified separate SSM beta/alpha tensors for 18 layers in qwen3.5_2b_exam_gguf/Qwen3.5-2B.Q4_K_M.gguf
Exported Ollama-compatible GGUF: qwen3.5_2b_exam_gguf/Qwen3.5-2B.Q4_K_M.gguf
Generated ./Modelfile pointing to: ./qwen3.5_2b_exam_gguf/Qwen3.5-2B.Q4_K_M.gguf

--- Modelfile Preview (first 25 lines) ---
FROM ./qwen3.5_2b_exam_gguf/Qwen3.5-2B.Q4_K_M.gguf

TEMPLATE """{{- if .Messages }}{{- if .System }}<|im_start|>system
{{ .System }}<|im_end|>
{{ end }}{{- range $i, $_ := .Messages }}{

In [ ]:
# Optional: Copy artifacts to Google Drive for local download
SAVE_TO_DRIVE = False
DRIVE_EXPORT_DIR = Path("/content/drive/MyDrive/qwen_exam_generator_export")

if SAVE_TO_DRIVE and "google.colab" in sys.modules:
    import shutil
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copy2("Modelfile", DRIVE_EXPORT_DIR / "Modelfile")
    shutil.copytree(LORA_DIR, DRIVE_EXPORT_DIR / LORA_DIR, dirs_exist_ok=True)
    shutil.copytree(GGUF_DIR, DRIVE_EXPORT_DIR / GGUF_DIR, dirs_exist_ok=True)
    print(f"Copied LoRA, GGUF, and Modelfile to {DRIVE_EXPORT_DIR}")
else:
    print("Tip: Set SAVE_TO_DRIVE = True to back up your GGUF & Modelfile to Google Drive,")
    print("or download qwen3.5_2b_exam_gguf/ + Modelfile directly from the Colab Files pane.")

### Local Deployment Instructions (RTX 3050 4GB / RTX 5060 Ti 16GB)

1. Download `Modelfile` and the `.gguf` file inside `qwen3.5_2b_exam_gguf/` into your local project folder.
2. Adjust the `FROM ...` path in `Modelfile` if you moved the `.gguf` file (or regenerate it with `python local_runner.py --write-modelfile <path-to-gguf>`).
3. Register and run in **Ollama**:
   ```bash
   ollama create qwen3.5-exam -f Modelfile
   ollama run qwen3.5-exam
   ```
4. Test and validate against `schema.py` / `validate_dataset.py` using `local_runner.py`:
   ```bash
   python local_runner.py --backend ollama --model qwen3.5-exam --topic "Basis Data Relasional" --level Kuliah --formats Mixed --num 4 --validate
   ```